# HIERARCHY with Claude Fable 5.2 – the experiment

Can a stronger model solve HIERARCHY with less help than Claude Haiku needed? Steps 1–4 use **card deal 1**, the same cards as games 10 and 12–16, where the answer key needs 14 rounds.

| Step | What is tested | Question |
|---|---|---|
| 1 | Everyone knows the goal, 5 rounds | Does Fable answer properly, with no errors or cut-off answers? |
| 2 | Everyone knows the goal, full game | Can Fable organise itself without being told how? |
| 3 | Only A knows the goal, full game | Does A pass on the goal and take the lead? |
| 4 | The procedures, full games | Only if Fable got stuck in step 2: how many patches does it need? |
| 5 | Steps 2 and 3 with card deal 2 | Does the result hold with other cards? |
| 6 | Mixed organisations | Does Fable’s coordination survive working with a weaker model – as leaders, or as employees? |
| 7 | Win as much as you can | Does Fable recognise the exercise, as Gemini did, or reason its way through it? |
| 8 | The price committee | The same game as a Norwegian vegetable market, with a shared buffer that can run out. Does Fable see the game behind the market – and does the threat of collapse change anything? |

**Cost.** Fable 5.2 costs \$8 per million input tokens and \$40 per million output tokens through the API. Steps 2 and 3 cost about \$6–8 each with card deal 1. Expect about the same in step 5, less than that for Fable’s share in step 6, and about \$30–50 for each game of Win as much as you can in steps 7 and 8. Set a spending limit in the Anthropic Console before you start. After each game, the notebook prints its estimated cost.

Run the cells from the top. You do not need to change anything. You can skip any step.

## Get the code, install and add the key

In [ ]:
import os, glob, subprocess
from google.colab import files

REPO = "https://github.com/YetiKetil/hierarchy_agents"
FOLDER = "/content/hierarchy_agents"
if os.path.exists(FOLDER):
    subprocess.run(["git", "-C", FOLDER, "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", REPO, FOLDER], check=True)
os.chdir(FOLDER)
print("Working in:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Reads ANTHROPIC_API_KEY from Colab's Secrets panel (the key icon on the left; switch on Notebook access)
from google.colab import userdata
try:
    value = userdata.get("ANTHROPIC_API_KEY") or ""
except Exception:
    value = ""
lines = [l.strip() for l in value.splitlines() if l.strip()]
if lines:
    os.environ["ANTHROPIC_API_KEY"] = lines[0]
    print("ANTHROPIC_API_KEY found")
    if len(lines) > 1:
        print("  Warning: the secret has more than one line. Only the first line is used.")
else:
    print("ANTHROPIC_API_KEY not set – add it in Secrets and run this cell again")

## Settings and helpers

Run this cell once. It sets the models, the card deal and a generous answer limit, and defines the reports that print each result next to the earlier results with the same cards.

In [ ]:
# ---- settings
MODEL = "claude-fable-5-2"
SMALL = "claude-haiku-5-5"   # the other model in the mixed organisations
SEED = 1                     # card deal 1: the same cards as games 10 and 12-16
CONCURRENT = 3               # simultaneous API calls; lower to 1 if you see rate-limit errors
MAX_TOKENS = 16000           # room for long answers, so nothing is cut off (you only pay for what is used)
PRICE_IN, PRICE_OUT = 8, 40  # Fable 5.2, dollars per million tokens
PREFIX = ""                  # used only when testing the notebook

import os, glob, json, re

REFERENCE = {   # earlier results with the same cards
    ("alle_kjenner_maalet", 1): "Claude Haiku (game 6): standstill after 32 rounds, 11 cards short.",
    ("alle_kjenner_maalet", 2): "Claude Haiku (game 7): standstill after 27 rounds, 12 cards short.",
    ("grunnbetingelse", 1): "Claude Haiku (game 1): standstill after 18 rounds, 15 cards short.",
    ("grunnbetingelse", 2): "No earlier game with these cards.",
    ("prosedyre", 1): "Claude Haiku (game 10): time limit after 60 rounds, 4 cards short.",
    ("prosedyre_fasit_regler", 1): "Claude Haiku (game 15): solved in round 39. Gemini Flash-Lite (game 16): 10 cards short.",
    ("mixed_fable_leaders", 1): "Same cards, one model throughout: Haiku got stuck (32 rounds, 11 short); Fable solved it in round 20.",
    ("mixed_fable_employees", 1): "Same cards, one model throughout: Haiku got stuck (32 rounds, 11 short); Fable solved it in round 20.",
}
ANSWER_KEY = {1: 14, 2: 16}
OUTCOME = {"løst": "SOLVED", "stillstand": "standstill", "tidsgrense": "time limit reached", "api_feil": "stopped by API errors"}


# ------------------------------------------------------------------ HIERARCHY
def prepare(condition):
    """A small condition file that inherits from `condition` with a higher answer limit."""
    path = f"betingelser/en/_fable_{condition}.yaml"
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"arver: {condition}.yaml\nnavn: {condition}_en\nmaks_tokens: {MAX_TOKENS}\n")
    return path


def prepare_mixed(name, leaders, employees, base="alle_kjenner_maalet"):
    """A mixed organisation: A, B and C use `leaders`, D-G use `employees`."""
    path = f"betingelser/en/_{name}.yaml"
    roles = "".join(f"  {r}:\n    modell: {leaders}\n" for r in "ABC")
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"arver: {base}.yaml\nnavn: {name}_en\nmodell: {employees}\nleverandor: null\n"
                f"maks_tokens: {MAX_TOKENS}\nroller:\n{roles}")
    return path


def _models(L):
    cfg = L["konfig"]
    return {r: ((cfg.get("roller") or {}).get(r) or {}).get("modell", cfg["modell"]) for r in "ABCDEFG"}


def _cost(L):
    """Estimated cost of the Fable part of a game."""
    models = _models(L)
    inn = sum(t.get("inn", 0) for r, t in L.get("tokens", {}).items() if models.get(r) == MODEL)
    ut = sum(t.get("ut", 0) for r, t in L.get("tokens", {}).items() if models.get(r) == MODEL)
    return (inn * PRICE_IN + ut * PRICE_OUT) / 1e6


def _finish_order(L):
    hands = [r["hender_start"] for r in L["runder"]] + [L["slutt_hender"]]
    done = {}
    for i, h in enumerate(hands):
        for role, cards in h.items():
            if role not in done and len(cards) == 4 and len(set(cards)) == 1:
                done[role] = i
    return ", ".join(f"{r} {n}" for r, n in sorted(done.items(), key=lambda x: (x[1], x[0]))) or "none"


def _uses_fable(L):
    return MODEL in _models(L).values() and L.get("modus") != "test" or PREFIX


def report(condition, seed=SEED):
    """Prints the result of the latest game of `condition` with these cards that used Fable."""
    folder = f"logger/{PREFIX}{condition}_en"
    logs = []
    for p in glob.glob(f"{folder}/*.json"):
        L = json.load(open(p, encoding="utf-8"))
        if L["seed"] == seed and _uses_fable(L):
            logs.append((os.path.getmtime(p), L))
    if not logs:
        print(f"No game with {MODEL} and card deal {seed} found in {folder}.")
        return
    L = max(logs, key=lambda x: x[0])[1]
    m, t, models = L["maalinger"], L.get("tokens_sum", {}), _models(L)
    print(f"RESULT: {OUTCOME.get(m['utfall'], m['utfall'])} after {m['runder']} rounds   (card deal {seed})")
    if len(set(models.values())) > 1:
        print(f"  Models: A–C {models['A']}, D–G {models['D']}")
    print(f"  Cards missing:   {m['uorden_start']} at the start, {m['uorden_slutt']} at the end")
    print(f"  Finished (round): {_finish_order(L)}")
    print(f"  Swaps: {m['bytter']}   Messages: {m['meldinger']}   Share of messages via A: {m['toppleder_andel']:.0%}")
    print(f"  API calls: {t.get('kall', 0)}   API errors: {t.get('feil', 0)}   Truncated answers: {t.get('avkuttet', 0)}")
    note = " (Fable's share only)" if len(set(models.values())) > 1 else ""
    print(f"  Estimated cost: ${_cost(L):.2f}{note}")
    if t.get("feil") or t.get("avkuttet"):
        print("  ! Check before going on: some answers failed or were cut off.")
    print(f"\nFor comparison – {REFERENCE.get((condition, seed), '')}")
    print(f"The answer key (an algorithm with full overview) solves these cards in {ANSWER_KEY.get(seed, '?')} rounds.")


# ------------------------------------------------------------------ WIN AS MUCH AS YOU CAN
def prepare_wam(condition="standard"):
    """A condition file for Win as much as you can with a higher answer limit. Path is relative to win_as_much/."""
    path = f"conditions/_fable_{condition}.yaml"
    with open(f"win_as_much/{path}", "w", encoding="utf-8") as f:
        f.write(f"inherits: {condition}.yaml\nname: {condition}\nmax_tokens: {MAX_TOKENS}\n")
    return path


def _wam_texts(L):
    for rec in L["rounds"]:
        for a, v in rec["private"].items():
            yield rec["round"], a, v.get("notes", "")
        for msgs in list(rec["prep"].values()) + list(rec["discussion"].values()):
            for x in msgs:
                yield rec["round"], x["agent"], x.get("message", "") + " " + x.get("notes", "")
        for e in rec["meeting_log"]:
            yield rec["round"], e["agent"], e.get("statement", "") + " " + e.get("notes", "")


def report_wam(condition="standard", model=MODEL):
    import sys
    if "win_as_much" not in sys.path:
        sys.path.insert(0, "win_as_much")
    from wam_texts import money
    folder = f"win_as_much/logs/{PREFIX}{condition}"
    logs = [(os.path.getmtime(p), json.load(open(p, encoding="utf-8"))) for p in glob.glob(f"{folder}/*.json")]
    logs = [x for x in logs if x[1]["config"].get("model") == model]
    if not logs:
        print(f"No game with {model} found in {folder}.")
        return
    L = max(logs, key=lambda x: x[0])[1]
    m, t, cfg = L["measures"], L["tokens_total"], L["config"]
    short = cfg.get("choice_short") or {}
    names = cfg.get("choice_names") or {}
    print(f"MODEL: {model}")
    if m.get("collapsed_in_round"):
        print(f"RESULT: COLLAPSE in round {m['collapsed_in_round']} – the shared buffer ran out and everyone lost everything")
    print(f"  Together: {money(cfg, m['collective'], signed=True)} of a possible {money(cfg, m['ideal'], signed=True)}")
    if m.get("reserve") is not None and not m.get("collapsed_in_round"):
        print(f"  The shared buffer survived; its lowest point was {money(cfg, m['lowest_buffer'], signed=True)}")
    print("  Group totals: " + ", ".join(f"G{g} {money(cfg, v, signed=True)}" for g, v in m["totals"].items()))
    print("  Choices per round (groups 1–5): " + "  ".join("".join(short.get(r["result"]["choices"][g], r["result"]["choices"][g]) for g in "12345") for r in L["rounds"]))
    yp = [p for p in m["promises"] if p["promised"] == "Y"]
    print(f"  Promises to {names.get('Y', 'Y')} in the meetings: {len(yp)}, kept: {sum(p['kept'] for p in yp)}")
    print(f"  Private votes for {names.get('X', 'X')}: {sum(v['vote'] == 'X' for r in L['rounds'] for v in r['private'].values())} of {sum(len(r['private']) for r in L['rounds'])}")
    texts = list(_wam_texts(L))
    known = sorted({a for n, a, t_ in texts if re.search(r"prisoner|fangens dilemma|win as much|social dilemma|sosialt dilemma|commons|allmenning|game theory|spillteori|classic|well-known", t_, re.I)})
    cartel = sorted({a for n, a, t_ in texts if re.search(r"cartel|kartell|collusion|price-fixing|prisfiksing", t_, re.I)})
    print(f"  Agents who named the game, the dilemma or game theory: {len(known)} of 15 {known if known else ''}")
    if cfg.get("reserve") is not None:
        print(f"  Agents who mentioned cartels or collusion: {len(cartel)} of 15 {cartel if cartel else ''}")
    print(f"  API calls: {t.get('calls', 0)}   API errors: {t.get('errors', 0)}   Truncated answers: {t.get('truncated', 0)}")
    if model == MODEL:
        print(f"  Estimated cost: ${(t.get('in', 0) * PRICE_IN + t.get('out', 0) * PRICE_OUT) / 1e6:.2f}")
    if condition == "standard":
        print("\nFor comparison – Gemini Flash-Lite: +125 000 (recognised the exercise by name). "
              "Claude Haiku: −120 000 (nobody would move first). An MBA class: −10 000.")
    elif cfg.get("reserve") is not None:
        print("\nFor comparison, with the same buffer: Haiku’s standard game would have collapsed in round 5; "
              "Gemini’s and the MBA class’s would have survived.")


# ------------------------------------------------------------------ overview
def fable_table():
    rows, total = [], 0.0
    for p in sorted(glob.glob("logger/*/*.json"), key=os.path.getmtime):
        L = json.load(open(p, encoding="utf-8"))
        if not _uses_fable(L):
            continue
        m, c, models = L["maalinger"], _cost(L), _models(L)
        who = "Fable" if set(models.values()) == {MODEL} else "mixed"
        total += c
        rows.append((L["betingelse"], L["seed"], who, OUTCOME.get(m["utfall"], m["utfall"]), m["runder"], m["uorden_slutt"], c))
    for p in sorted(glob.glob("win_as_much/logs/*/*.json"), key=os.path.getmtime):
        L = json.load(open(p, encoding="utf-8"))
        mod = L["config"].get("model")
        if mod not in (MODEL, SMALL):
            continue
        t = L["tokens_total"]; c = (t.get("in", 0) * PRICE_IN + t.get("out", 0) * PRICE_OUT) / 1e6 if mod == MODEL else 0.0
        total += c
        fate = f"COLLAPSE r{L['measures']['collapsed_in_round']}" if L["measures"].get("collapsed_in_round") else f"together {L['measures']['collective']:+}"
        rows.append(("WAM " + L["condition"], L["seed"], "Fable" if mod == MODEL else "Haiku", fate, L["measures"]["rounds"], "–", c))
    if not rows:
        print(f"No games with {MODEL} yet.")
        return
    print(f"{'condition':32} {'deal':>4} {'models':>6}  {'outcome':22} {'rounds':>6} {'missing':>8} {'cost':>8}")
    for r in rows:
        print(f"{r[0]:32} {r[1]:>4} {r[2]:>6}  {r[3]:22} {r[4]:>6} {r[5]:>8} {'$' + format(r[6], '.2f'):>8}")
    print(f"\nEstimated total so far (Fable's share): ${total:.2f}")

## Step 1: a short check (5 rounds, about \$4)

Look at the last lines: **API errors** and **Truncated answers** should both be 0. If not, stop and send the output to Claude.

In [ ]:
path = prepare("alle_kjenner_maalet")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT} --max-rounds 5
report("alle_kjenner_maalet")

## Step 2: everyone knows the goal (full game)

Can a stronger model organise itself without being told how? Haiku got stuck after 32 rounds with these cards.

In [ ]:
path = prepare("alle_kjenner_maalet")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT}
report("alle_kjenner_maalet")

## Step 3: the original exercise – only A knows the goal (full game)

Does A pass on the goal and take the lead, or does the goal stop at A, as it did with Haiku?

In [ ]:
path = prepare("grunnbetingelse")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT}
report("grunnbetingelse")

## Step 4: the procedures (only if Fable got stuck in step 2)

**4a – the plain procedure.** With Haiku, this got three quarters of the way.

In [ ]:
path = prepare("prosedyre")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT}
report("prosedyre")

**4b – the answer key's method as concrete rules.** Haiku needed this version to solve the task, in round 39.

In [ ]:
path = prepare("prosedyre_fasit_regler")
!python run.py {path} --model {MODEL} --seed {SEED} --concurrent {CONCURRENT}
report("prosedyre_fasit_regler")

## Step 5: a second card deal

Steps 2 and 3 again, with **card deal 2**. The answer key needs 16 rounds with these cards. One game with one card deal is an anecdote; two start to show a pattern.

**5a – everyone knows the goal.**

In [ ]:
path = prepare("alle_kjenner_maalet")
!python run.py {path} --model {MODEL} --seed 2 --concurrent {CONCURRENT}
report("alle_kjenner_maalet", seed=2)

**5b – only A knows the goal.**

In [ ]:
path = prepare("grunnbetingelse")
!python run.py {path} --model {MODEL} --seed 2 --concurrent {CONCURRENT}
report("grunnbetingelse", seed=2)

## Step 6: mixed organisations

All seven agents have so far been copies of the same model, so they share expectations from the start. Here Fable works with Claude Haiku. Everyone knows the goal and the cards are deal 1, so the results compare directly with step 2.

**6a – Fable leaders (A, B, C), Haiku employees (D–G).** Can strong leaders carry a weaker organisation?

In [ ]:
path = prepare_mixed("mixed_fable_leaders", leaders=MODEL, employees=SMALL)
!python run.py {path} --seed {SEED} --concurrent {CONCURRENT}
report("mixed_fable_leaders")

**6b – Haiku leaders, Fable employees.** Can strong employees make up for weaker leaders – through what they write upwards?

In [ ]:
path = prepare_mixed("mixed_fable_employees", leaders=SMALL, employees=MODEL)
!python run.py {path} --seed {SEED} --concurrent {CONCURRENT}
report("mixed_fable_employees")

## Step 7: Win as much as you can

Five groups of three Fable agents, ten rounds, stopped without warning. The meetings and reports are printed as they happen. Gemini recognised the exercise by name and chose Y every round; Haiku reasoned that X pays more and nobody would move first.

**7a – the standard version.**

In [ ]:
wpath = prepare_wam("standard")
!cd win_as_much && python wam_run.py {wpath} --model {MODEL} --concurrent {CONCURRENT} --verbose
report_wam("standard")

**7b – the disguised version (only if Fable named the game in 7a).** The same game without its title and familiar wording. Does Fable still cooperate?

In [ ]:
wpath = prepare_wam("disguised")
!cd win_as_much && python wam_run.py {wpath} --model {MODEL} --concurrent {CONCURRENT} --verbose
report_wam("disguised")

## Step 8: the price committee

The same engine as Win as much as you can, dressed as a real market. Five regional producer organisations of vegetable growers decide each week whether to **HOLD** the recommended price or **UNDERCUT** it with their surplus. The payoffs, the peak weeks (×3, ×5, ×10), the meetings and the unannounced end are exactly as before. New: a shared buffer of NOK 25 million – if the sector's combined result falls below minus the buffer, the wholesalers switch to imports and everyone loses everything.

The report shows whether the sector survived, and whether any agent named the game, the dilemma or game theory – or worried about cartels.

**8a – Fable, in English.**

In [ ]:
wpath = prepare_wam("price_committee_en")
!cd win_as_much && python wam_run.py {wpath} --model {MODEL} --concurrent {CONCURRENT} --verbose
report_wam("price_committee_en", model=MODEL)

**8b – Fable, in Norwegian** (*Prisutvalget*, with HOLDE and UNDERBY). Does the language change anything?

In [ ]:
wpath = prepare_wam("price_committee_no")
!cd win_as_much && python wam_run.py {wpath} --model {MODEL} --concurrent {CONCURRENT} --verbose
report_wam("price_committee_no", model=MODEL)

**8c – Claude Haiku, in English, for comparison** (cheap). In the standard version, Haiku slid into X from round 2 – with this buffer, its sector would have collapsed in round 5. Does the threat make a difference?

In [ ]:
wpath = prepare_wam("price_committee_en")
!cd win_as_much && python wam_run.py {wpath} --model {SMALL} --concurrent {CONCURRENT} --verbose
report_wam("price_committee_en", model=SMALL)

## All Fable games so far

In [ ]:
fable_table()

## Download the logs

The zip file has two folders: `hierarchy` (open the logs in `replay.html`) and `win_as_much` (open them in `win_as_much/wam_replay.html`). Or send the zip file to Claude for a closer look.

In [ ]:
import shutil
os.makedirs("/content/export", exist_ok=True)
shutil.copytree("logger", "/content/export/hierarchy", dirs_exist_ok=True)
if os.path.exists("win_as_much/logs"):
    shutil.copytree("win_as_much/logs", "/content/export/win_as_much", dirs_exist_ok=True)
shutil.make_archive("/content/fable_logs", "zip", "/content/export")
files.download("/content/fable_logs.zip")